# 04 | Candidate-generation experiments

Radius/cap sensitivity on a training trip. No student-test or evaluator outcomes are used to choose the defaults in this experiment.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/data/dataset_loader.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PathSnap repository or notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = Path(os.environ.get("PATHSNAP_RAW_ROOT", str(ROOT / "data/raw")))
EVIDENCE = ROOT / "results/member1"
if not (EVIDENCE / "processing_summary.csv").exists():
    raise RuntimeError("Run python -m src.data.prepare_member1 first")
inventory = pd.read_csv(EVIDENCE / "dataset_inventory.csv", dtype={"trip_id": str})
summary = pd.read_csv(EVIDENCE / "processing_summary.csv", dtype={"trip_id": str})
display(inventory.groupby("partition").agg(trips=("trip_id", "size"), gps=("gps_observations", "sum")))

,trips,gps
partition,,
student_test,10,19160
train,80,208417


In [2]:
from src.data.dataset_loader import load_trip
from src.data.prepare_member1 import prepare_trip, candidate_experiment
row = inventory.loc[inventory.partition == "train"].iloc[0]
trip = load_trip(RAW, row.partition, row.trip_id, include_reference=True)
track, rejected, candidates, features, labels, network = prepare_trip(trip)
experiment = candidate_experiment(track, network, labels)
display(experiment)
display(summary.groupby("partition")[["clean_observations", "candidate_rows", "fallback_observations", "usable_proxy_labels", "candidate_misses"]].sum())

,radius_m,max_candidates,rows,usable_proxy_observations,candidate_miss_rate,fallback_observations,elapsed_s
0,25.0,4,4208,1104,0.009058,0,0.089542
1,25.0,8,5722,1104,0.000906,0,0.088966
2,25.0,12,6221,1104,0.000906,0,0.092412
3,50.0,4,4958,1104,0.009058,0,0.100254
4,50.0,8,8698,1104,0.000906,0,0.098880
5,50.0,12,11292,1104,0.000000,0,0.101359
6,100.0,4,5205,1104,0.009058,0,0.131160
7,100.0,8,10062,1104,0.000906,0,0.141683
8,100.0,12,14507,1104,0.000000,0,0.142981


,clean_observations,candidate_rows,fallback_observations,usable_proxy_labels,candidate_misses
partition,,,,,
student_test,19160,113648,0,14472,15
train,208417,1124837,18230,145069,726


In [3]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for cap, group in experiment.groupby("max_candidates"):
    axes[0].plot(group.radius_m, group.candidate_miss_rate, marker="o", label=f"cap={cap}")
    axes[1].plot(group.radius_m, group.rows / len(track), marker="o", label=f"cap={cap}")
axes[0].set(xlabel="Radius (m)", ylabel="Miss rate among usable proxies", title="Reference proxy coverage")
axes[1].set(xlabel="Radius (m)", ylabel="Mean candidate rows / observation", title="Feature-table expansion")
axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()
assert not candidates.duplicated(["observation_id", "candidate_arc_id"]).any()
print("Default candidate cap:", candidates.groupby("observation_id").size().max())

Default candidate cap: 8


C:\Users\nehas\AppData\Local\Temp\ipykernel_17884\3865341388.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()


## Interpretation
Coverage is measured against approximate aligned reference labels; excluded-label counts must accompany miss rates. Do not inject a reference candidate to improve coverage. Nearest fallbacks remain flagged even if far away. This one-trip sensitivity study is not an all-trip policy optimization; Member 2 should validate settings using trip-grouped experiments.